# Notebook 06 — Effect of aperiodic correction on oscillatory findings

Generates aperiodic-corrected ("flattened") spectra and re-runs peak detection.  
Compares oscillatory summary statistics **before** and **after** correction to  
identify which regional oscillatory findings are stable vs. artifacts of the  
aperiodic background.

**Inputs**:
- `data/interim/specparam_ieeg_results.csv` — full results with peaks (from NB 02)
- Raw PSDs recomputed from the MATLAB file (same as NB 02)

**Output**: Figure 5 — Before vs. after aperiodic correction

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import scipy.io
import h5py
from scipy.signal import welch
import matplotlib.pyplot as plt
from specparam import SpectralGroupModel

PROJECT_ROOT = Path("../../").resolve()
INTERIM_DIR  = PROJECT_ROOT / "data" / "interim"
FIG_DIR      = PROJECT_ROOT / "manuscript" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

FREQ_RANGE        = (2.0, 40.0)
APERIODIC_MODE    = "fixed"
PEAK_WIDTH_LIMITS = [1.0, 12.0]
MAX_N_PEAKS       = 6
MIN_PEAK_HEIGHT   = 0.05
PEAK_THRESHOLD    = 2.0
R2_THRESHOLD      = 0.80

FS = 200.0

## 1. Load iEEG results and raw PSDs

Load the channel-level specparam results from NB 02, and recompute (or reload)  
the raw Welch PSDs to reconstruct the aperiodic curve for each channel.

In [ ]:
# Load specparam results (includes peaks AND aperiodic params)
results_full = pd.read_csv(INTERIM_DIR / "specparam_ieeg_results.csv")
# Per-channel summary (one row per channel)
ch_df = results_full.drop_duplicates(subset="ID")[[
    "ID", "offset", "exponent", "gof_rsquared", "good_fit", "Region name", "Lobe"
]].copy()

print(f"Loaded {len(ch_df)} channels, {results_full.dropna(subset=['CF']).shape[0]} peaks")

In [ ]:
# Recompute raw PSDs (same as NB 02)
IEEG_MAT = PROJECT_ROOT / "data" / "Frauscher2018" / "WakefulnessMatlabFile.mat"

print("Loading data …")
try:
    mat = scipy.io.loadmat(str(IEEG_MAT))
    data = mat["Data"].T
except Exception:
    with h5py.File(IEEG_MAT, "r") as f:
        data = np.array(f["Data"]).T

freqs, psds = welch(
    data, fs=FS, nperseg=int(2*FS), noverlap=int(FS),
    detrend="constant", scaling="density"
)
print(f"PSDs: {psds.shape}")

## 2. Reconstruct aperiodic curves and compute corrected spectra

The aperiodic component in specparam's fixed mode is:  
`L(f) = offset - log10(f^exponent) = offset - exponent * log10(f)`  
In linear power units: `P_ap(f) = 10^offset / f^exponent`

The **corrected (flattened) spectrum** in log units is:  
`corrected(f) = log10(P(f)) - L(f)`

In [ ]:
# Crop to fitting range
freq_mask  = (freqs >= FREQ_RANGE[0]) & (freqs <= FREQ_RANGE[1])
freqs_fit  = freqs[freq_mask]
psds_fit   = psds[:, freq_mask]


def aperiodic_curve_log(freqs, offset, exponent):
    """Fitted aperiodic in log10(power) units."""
    return offset - exponent * np.log10(freqs)


# Compute corrected spectra for good-fit channels
good_ids   = ch_df[ch_df["good_fit"] == True]["ID"].values
corrected  = np.full(psds_fit.shape, np.nan)

# Get aperiodic fit params from group_results (rc6: use aperiodic_fit, not aperiodic_converted)
group_res = fg.results.group_results

for idx in good_ids:
    ap_fit   = group_res[idx].aperiodic_fit  # [offset, exponent] for fixed mode
    offset   = ap_fit[0]
    exponent = ap_fit[-1]
    ap_log   = aperiodic_curve_log(freqs_fit, offset, exponent)
    corrected[idx, :] = np.log10(psds_fit[idx, :] + 1e-30) - ap_log

print(f"Corrected spectra computed for {len(good_ids)} channels")

## 3. Peak detection on corrected spectra

Re-fit specparam on the **corrected** spectra with `aperiodic_mode='fixed'`.  
Since the 1/f background has been removed, the corrected spectra should be nearly  
flat, so we expect a lower exponent and peaks (if any) corresponding only to  
true oscillatory bumps.

In [ ]:
def specparam2pandas(fg):
    """
    Converts a SpectralGroupModel object into a pandas DataFrame, with peak parameters and
    corresponding aperiodic fit information.

    Returns a long-format DataFrame where each row represents a single peak.
    Spectra with no peaks are retained with NaN values for peak columns.
    Columns: CF, PW, BW, ID, offset, exponent (+ knee if knee mode), error_mae, gof_rsquared.
    """
    if not fg.results.has_model:
        raise ValueError("No model fit results available. Please fit the model first.")

    ap_params = fg.get_params("aperiodic")
    ap_labels = list(fg.modes.aperiodic.params.labels)

    specparam_aperiodic = pd.DataFrame(ap_params, columns=ap_labels)

    # Direct dict access — fg.get_metrics() silently returns NaN in some rc6 builds
    group_res = fg.results.group_results
    specparam_aperiodic["error_mae"]    = [r.metrics.get("error_mae",    np.nan) for r in group_res]
    specparam_aperiodic["gof_rsquared"] = [r.metrics.get("gof_rsquared", np.nan) for r in group_res]
    specparam_aperiodic = specparam_aperiodic.reset_index(names=["ID"])

    peaks = fg.get_params("peak")

    if peaks.size > 0:
        peak_df = pd.DataFrame(peaks, columns=["CF", "PW", "BW", "ID"])
        peak_df["ID"] = peak_df["ID"].astype(int)
        result = specparam_aperiodic.merge(peak_df, on="ID", how="left")
    else:
        peak_df = pd.DataFrame(columns=["CF", "PW", "BW", "ID"])
        result = specparam_aperiodic.merge(peak_df, on="ID", how="left")

    return result

## 4. Compare peaks before vs. after correction

**Before**: Peaks from NB 02 (fit on raw log-PSD — peaks reflect oscillatory bumps  
above the 1/f background that specparam estimated).

**After**: Peaks from the corrected spectra above.  

We compare: (a) peak prevalence per region, (b) peak centre frequencies.

In [ ]:
# Peaks BEFORE correction (from NB 02 results)
peaks_before = results_full.dropna(subset=["CF"]).copy()
peaks_before = peaks_before[peaks_before["good_fit"] == True]

# Peaks AFTER correction (mapped back to channel_ID → region)
ch_region_map = ch_df[["ID", "Region name", "Lobe"]].set_index("ID")

peaks_after = peaks_corrected.dropna(subset=["CF"]).copy()
peaks_after["Region name"] = peaks_after["channel_ID"].map(ch_region_map["Region name"])
peaks_after["Lobe"]        = peaks_after["channel_ID"].map(ch_region_map["Lobe"])

print(f"Peaks BEFORE correction: {len(peaks_before)}")
print(f"Peaks AFTER  correction: {len(peaks_after)}")

In [ ]:
# Band labelling helper
def assign_band(cf):
    if cf < 4:    return "delta"
    elif cf < 8:  return "theta"
    elif cf < 13: return "alpha"
    elif cf < 30: return "beta"
    else:         return "gamma"

peaks_before["band"] = peaks_before["CF"].apply(assign_band)
peaks_after["band"]  = peaks_after["CF"].apply(assign_band)

# Band count before/after
band_before = peaks_before["band"].value_counts().rename("before")
band_after  = peaks_after["band"].value_counts().rename("after")
band_comp   = pd.concat([band_before, band_after], axis=1).fillna(0).astype(int)
band_comp["change"] = band_comp["after"] - band_comp["before"]
print("\nPeak counts by band:")
display(band_comp)

## 5. Figure 5 — Before vs. after correction

In [ ]:
# Region-level peak prevalence: fraction of channels with at least one peak
def region_peak_prevalence(peaks_df, ch_summary, band=None):
    df = peaks_df.copy()
    if band:
        df = df[df["band"] == band]
    has_peak = df.drop_duplicates(subset="ID" if "ID" in df else "channel_ID").groupby("Region name").size()
    n_ch_per_region = ch_summary[ch_summary["good_fit"] == True].groupby("Region name").size()
    prevalence = (has_peak / n_ch_per_region).fillna(0)
    return prevalence


ch_summary = pd.read_csv(INTERIM_DIR / "specparam_ieeg_ch_summary.csv")

prev_before = region_peak_prevalence(peaks_before, ch_summary)
prev_after  = region_peak_prevalence(
    peaks_after.rename(columns={"channel_ID": "ID"}),
    ch_summary
)

# Combine
compare_df = pd.DataFrame({"before": prev_before, "after": prev_after}).dropna()
compare_df["delta"] = compare_df["after"] - compare_df["before"]
compare_df = compare_df.sort_values("delta")

fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# Left: scatter before vs after
ax = axes[0]
ax.scatter(compare_df["before"], compare_df["after"], s=50, alpha=0.7)
lims = [0, 1]
ax.plot(lims, lims, "k--", alpha=0.5, label="no change")
for region in compare_df[compare_df["delta"].abs() > 0.15].index:
    ax.annotate(region,
                (compare_df.loc[region, "before"], compare_df.loc[region, "after"]),
                fontsize=6, xytext=(3, 3), textcoords="offset points")
ax.set_xlabel("Peak prevalence — raw spectrum")
ax.set_ylabel("Peak prevalence — corrected spectrum")
ax.set_title("Regional peak prevalence: before vs. after aperiodic correction")
ax.legend()

# Right: change bar chart (which regions gain/lose peaks)
ax = axes[1]
y  = np.arange(len(compare_df))
colors = ["#E64B35" if d < 0 else "#00A087" for d in compare_df["delta"]]
ax.barh(y, compare_df["delta"], color=colors, edgecolor="white")
ax.set_yticks(y)
ax.set_yticklabels(compare_df.index, fontsize=7)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("Δ prevalence (after − before)")
ax.set_title("Change in peak prevalence after aperiodic correction")

fig.suptitle("Figure 5 — Effect of aperiodic correction on oscillatory findings", fontsize=12)
plt.tight_layout()
plt.savefig(FIG_DIR / "fig5_aperiodic_correction_effects.svg", bbox_inches="tight", dpi=150)
plt.show()

## 6. Example: corrected vs. raw spectrum for selected regions

In [ ]:
# Show one raw + corrected spectrum pair per lobe (median channel by exponent)
lobes_to_show = ["Frontal", "Temporal", "Occipital"]

fig, axes = plt.subplots(1, len(lobes_to_show), figsize=(5 * len(lobes_to_show), 5))

for ax, lobe in zip(axes, lobes_to_show):
    lobe_ids = ch_df[(ch_df["Lobe"] == lobe) & (ch_df["good_fit"] == True)]["ID"].values
    if len(lobe_ids) == 0:
        continue
    # Pick the channel closest to median exponent for this lobe
    lobe_exp  = ch_df.set_index("ID").loc[lobe_ids, "exponent"]
    median_ch = lobe_exp.sub(lobe_exp.median()).abs().idxmin()

    row      = ch_df[ch_df["ID"] == median_ch].iloc[0]
    raw_log  = np.log10(psds_fit[median_ch, :])
    ap_log   = aperiodic_curve_log(freqs_fit, row["offset"], row["exponent"])
    corr_log = raw_log - ap_log

    ax.plot(freqs_fit, raw_log,  label="raw (log10)", color="steelblue")
    ax.plot(freqs_fit, ap_log,   label="aperiodic fit", linestyle="--", color="steelblue", alpha=0.6)
    ax.plot(freqs_fit, corr_log, label="corrected", color="coral")
    ax.axhline(0, color="black", linewidth=0.5, linestyle=":")
    ax.set_xlabel("Frequency (Hz)")
    ax.set_ylabel("log₁₀ PSD")
    ax.set_title(f"{lobe} — exp={row['exponent']:.2f}")
    ax.legend(fontsize=7)

plt.suptitle("Raw vs. corrected spectra by lobe", y=1.02)
plt.tight_layout()
plt.show()

## 7. Summary: stable vs. changed oscillatory findings

In [ ]:
threshold = 0.10  # >10 percentage points change = meaningful shift
stable    = compare_df[compare_df["delta"].abs() <= threshold]
changed   = compare_df[compare_df["delta"].abs() >  threshold]

print(f"Regions with STABLE peak prevalence (|Δ| ≤ {threshold}): {len(stable)}")
display(stable.sort_values("before", ascending=False).head(10).round(3))

print(f"\nRegions with CHANGED peak prevalence (|Δ| > {threshold}): {len(changed)}")
display(changed.sort_values("delta").round(3))

compare_df.to_csv(INTERIM_DIR / "peak_prevalence_before_after_correction.csv")
print("\nSaved comparison table.")